# Fully Sharded Data Parallel (FSDP) Tutorial

## Overview

FSDP shards model parameters, gradients, and optimizer states across GPUs, enabling training of models larger than single GPU memory.

### Learning Objectives
- Understand FSDP sharding strategies
- Implement memory-efficient training
- Compare FSDP vs DDP memory usage
- Master FSDP configuration options

### References
- Zhao et al., "PyTorch FSDP: Experiences on Scaling Fully Sharded Data Parallel", VLDB 2023
- [PyTorch FSDP Tutorial](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)

## 1. Mathematical Foundation

### 1.1 Memory Analysis

For a model with $\Psi$ parameters using mixed precision:

**DDP Memory per GPU:**
$$M_{\text{DDP}} = 2\Psi + 2\Psi + (4\Psi + 4\Psi + 4\Psi) = 16\Psi$$

- Parameters (FP16): $2\Psi$
- Gradients (FP16): $2\Psi$
- Optimizer states (FP32): $12\Psi$ (Adam: params + momentum + variance)

**FSDP Memory per GPU (N GPUs):**
$$M_{\text{FSDP}} = \frac{16\Psi}{N} + 2\Psi_{\text{active}}$$

Where $\Psi_{\text{active}}$ is the size of currently active (unsharded) layer.

### 1.2 Communication Cost

FSDP requires AllGather before forward/backward and ReduceScatter after backward:

$$T_{\text{FSDP}} = 3 \times \frac{(N-1)}{N} \times \Psi \times (\alpha + \beta)$$

vs DDP:
$$T_{\text{DDP}} = 2 \times \frac{(N-1)}{N} \times \Psi \times (\alpha + \beta)$$

FSDP has ~1.5x communication overhead but enables much larger models.

## 2. Environment Setup

In [ ]:
import os
import functools
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
from torch.utils.data import DataLoader, DistributedSampler

print(f"PyTorch version: {torch.__version__}")
print(f"FSDP available: {hasattr(torch.distributed, 'fsdp')}")

## 3. FSDP Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                      FSDP Sharding Flow                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Initial State (Sharded):                                       │
│  GPU 0: [P0]    GPU 1: [P1]    GPU 2: [P2]    GPU 3: [P3]      │
│                                                                 │
│  Forward Pass:                                                  │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ AllGather: Reconstruct full parameters for computation  │   │
│  │ GPU 0: [P0,P1,P2,P3] → compute → discard non-local     │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  Backward Pass:                                                 │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │ AllGather: Reconstruct params for gradient computation  │   │
│  │ Compute local gradients                                 │   │
│  │ ReduceScatter: Aggregate and shard gradients           │   │
│  └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│  Final State (Sharded):                                         │
│  GPU 0: [G0]    GPU 1: [G1]    GPU 2: [G2]    GPU 3: [G3]      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 4. Sharding Strategies

In [ ]:
# FSDP Sharding Strategies
sharding_strategies = {
    "FULL_SHARD": ShardingStrategy.FULL_SHARD,      # Shard params, grads, optimizer
    "SHARD_GRAD_OP": ShardingStrategy.SHARD_GRAD_OP, # Shard grads and optimizer only
    "NO_SHARD": ShardingStrategy.NO_SHARD,          # Like DDP, no sharding
    "HYBRID_SHARD": ShardingStrategy.HYBRID_SHARD,  # Shard within node, replicate across
}

print("Sharding Strategy Comparison:")
print("="*60)
print(f"{'Strategy':<20} {'Memory':<15} {'Communication':<15}")
print("-"*60)
print(f"{'FULL_SHARD':<20} {'Lowest':<15} {'Highest':<15}")
print(f"{'SHARD_GRAD_OP':<20} {'Medium':<15} {'Medium':<15}")
print(f"{'HYBRID_SHARD':<20} {'Medium':<15} {'Low (intra-node)':<15}")
print(f"{'NO_SHARD':<20} {'Highest':<15} {'Lowest':<15}")

## 5. Basic FSDP Implementation

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer block for FSDP wrapping demonstration."""
    
    def __init__(self, d_model: int = 512, nhead: int = 8, dim_ff: int = 2048):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_ff),
            nn.GELU(),
            nn.Linear(dim_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x


class SimpleTransformer(nn.Module):
    """Simple Transformer model for FSDP demonstration."""
    
    def __init__(self, vocab_size: int = 10000, d_model: int = 512, 
                 num_layers: int = 6, nhead: int = 8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, nhead) for _ in range(num_layers)
        ])
        self.output = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return self.output(x)

In [ ]:
def create_fsdp_model(model: nn.Module, rank: int) -> FSDP:
    """Wrap model with FSDP.
    
    Args:
        model: PyTorch model
        rank: Process rank
        
    Returns:
        FSDP-wrapped model
    """
    # Auto-wrap policy: wrap each TransformerBlock separately
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerBlock}
    )
    
    # Mixed precision configuration
    mixed_precision = MixedPrecision(
        param_dtype=torch.float16,
        reduce_dtype=torch.float16,
        buffer_dtype=torch.float16
    )
    
    fsdp_model = FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        auto_wrap_policy=auto_wrap_policy,
        mixed_precision=mixed_precision,
        device_id=rank,
        use_orig_params=True,  # Required for torch.compile
    )
    
    return fsdp_model

## 6. Memory Comparison: DDP vs FSDP

In [ ]:
def estimate_memory_usage(num_params: int, num_gpus: int, precision: str = "fp16"):
    """Estimate memory usage for DDP vs FSDP.
    
    Args:
        num_params: Number of model parameters
        num_gpus: Number of GPUs
        precision: "fp16" or "fp32"
    """
    bytes_per_param = 2 if precision == "fp16" else 4
    
    # DDP: Full model + gradients + optimizer states on each GPU
    ddp_params = num_params * bytes_per_param
    ddp_grads = num_params * bytes_per_param
    ddp_optimizer = num_params * 12  # Adam: 3 * 4 bytes (FP32)
    ddp_total = ddp_params + ddp_grads + ddp_optimizer
    
    # FSDP: Sharded across GPUs
    fsdp_sharded = (ddp_params + ddp_grads + ddp_optimizer) / num_gpus
    fsdp_active = ddp_params  # One layer unsharded during compute
    fsdp_total = fsdp_sharded + fsdp_active * 0.1  # ~10% for active layer
    
    print(f"Model: {num_params/1e9:.2f}B parameters, {num_gpus} GPUs")
    print(f"\nDDP Memory per GPU:")
    print(f"  Parameters: {ddp_params/1e9:.2f} GB")
    print(f"  Gradients:  {ddp_grads/1e9:.2f} GB")
    print(f"  Optimizer:  {ddp_optimizer/1e9:.2f} GB")
    print(f"  Total:      {ddp_total/1e9:.2f} GB")
    print(f"\nFSDP Memory per GPU:")
    print(f"  Sharded:    {fsdp_sharded/1e9:.2f} GB")
    print(f"  Active:     {fsdp_active*0.1/1e9:.2f} GB")
    print(f"  Total:      {fsdp_total/1e9:.2f} GB")
    print(f"\nMemory Reduction: {(1 - fsdp_total/ddp_total)*100:.1f}%")

# Example: 7B parameter model on 8 GPUs
estimate_memory_usage(7_000_000_000, 8, "fp16")

## 7. FSDP Training Loop

In [ ]:
def fsdp_training_worker(rank: int, world_size: int, epochs: int = 3):
    """FSDP training worker function."""
    # Setup distributed
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    dist.init_process_group('nccl', rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    
    # Create model and wrap with FSDP
    model = SimpleTransformer(vocab_size=10000, d_model=512, num_layers=6)
    fsdp_model = create_fsdp_model(model, rank)
    
    # Synthetic data
    dataset = torch.utils.data.TensorDataset(
        torch.randint(0, 10000, (500, 128)),  # Input tokens
        torch.randint(0, 10000, (500, 128))   # Target tokens
    )
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, batch_size=8, sampler=sampler)
    
    optimizer = torch.optim.AdamW(fsdp_model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        fsdp_model.train()
        total_loss = 0
        
        for batch_idx, (data, target) in enumerate(dataloader):
            data, target = data.to(rank), target.to(rank)
            
            optimizer.zero_grad()
            output = fsdp_model(data)
            loss = criterion(output.view(-1, 10000), target.view(-1))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if rank == 0:
            print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")
    
    dist.destroy_process_group()

## 8. Checkpointing with FSDP

In [ ]:
from torch.distributed.fsdp import StateDictType, FullStateDictConfig

def save_fsdp_checkpoint(model: FSDP, optimizer, path: str, rank: int):
    """Save FSDP checkpoint with full state dict."""
    # Configure to gather full state dict on rank 0
    full_state_config = FullStateDictConfig(offload_to_cpu=True, rank0_only=True)
    
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, full_state_config):
        state_dict = model.state_dict()
        optim_state = FSDP.optim_state_dict(model, optimizer)
        
        if rank == 0:
            torch.save({
                'model': state_dict,
                'optimizer': optim_state
            }, path)
            print(f"Checkpoint saved to {path}")


def load_fsdp_checkpoint(model: FSDP, optimizer, path: str):
    """Load FSDP checkpoint."""
    checkpoint = torch.load(path, map_location='cpu')
    
    with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT):
        model.load_state_dict(checkpoint['model'])
        optim_state = FSDP.optim_state_dict_to_load(
            model, optimizer, checkpoint['optimizer']
        )
        optimizer.load_state_dict(optim_state)
    
    print(f"Checkpoint loaded from {path}")

## 9. Advanced: CPU Offloading

In [ ]:
from torch.distributed.fsdp import CPUOffload

def create_fsdp_with_offload(model: nn.Module, rank: int) -> FSDP:
    """Create FSDP model with CPU offloading for extreme memory savings."""
    
    auto_wrap_policy = functools.partial(
        transformer_auto_wrap_policy,
        transformer_layer_cls={TransformerBlock}
    )
    
    fsdp_model = FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        auto_wrap_policy=auto_wrap_policy,
        cpu_offload=CPUOffload(offload_params=True),  # Offload to CPU
        device_id=rank,
    )
    
    return fsdp_model

print("CPU Offload Trade-offs:")
print("+ Enables training models 2-3x larger than GPU memory")
print("+ Reduces GPU memory to near-zero when idle")
print("- Significant slowdown (2-5x) due to CPU-GPU transfers")
print("- Requires fast CPU memory bandwidth")

## 10. Summary

### Key Takeaways

1. **FSDP Sharding**: Distributes params, grads, optimizer across GPUs
2. **Memory Efficiency**: ~N times reduction with N GPUs
3. **Communication**: AllGather + ReduceScatter pattern
4. **Auto-wrap Policy**: Wrap transformer layers for optimal sharding
5. **CPU Offload**: Further memory reduction at speed cost

### When to Use FSDP

| Scenario | Recommendation |
|----------|----------------|
| Model fits in GPU | Use DDP (faster) |
| Model slightly exceeds GPU | Use SHARD_GRAD_OP |
| Model far exceeds GPU | Use FULL_SHARD |
| Extreme memory constraints | Use FULL_SHARD + CPU Offload |